# LLM Fine-tuning: PEFT, LoRA & QLoRA

Reach for this when you need: 
- Reference for fine-tuning Billion-parameter models on consumer hardware.
- To implement Parameter-Efficient Fine-Tuning (PEFT).
- Understanding Low-Rank Adaptation (LoRA).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. QLoRA: 4-bit Quantization

Quantization reduces the memory footprint of weights (e.g. 16-bit to 4-bit) while keeping performance high through 'Double Quantization'.

| Param | Description | Industry Usage |
| :--- | :--- | :--- |
| `load_in_4bit` | Uses NF4 quantization | Essential for Llama/Mistral on single GPUs |
| `bnb_4bit_compute_dtype` | Intermediate math dtype | Usually `torch.bfloat16` for speed |
| `bnb_4bit_use_double_quant` | Extra compression | Saves additional VRAM |

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto"
)

## 2. LoRA (Low-Rank Adaptation)

Instead of updating all weights, inject small rank-decomposition matrices into layers. Only these small matrices are trained.

✅ **Use when**: Adapting LLMs to specific styles or instruction datasets with limited VRAM.
❌ **Don't use when**: You need to significantly change the model's fundamental knowledge baseline.

In [ ]:
config = LoraConfig(
    r=16, # Rank: smaller = less params, larger = more capacity
    lora_alpha=32, # Scaling factor
    target_modules=["q_proj", "v_proj"], # Modules to apply LoRA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
peft_model = get_peft_model(model, config)

peft_model.print_trainable_parameters() # Usually < 1% of total params

### Common Pitfalls
- **Target Modules**: Each architecture has different module names (e.g. `query_key_value` in GPT-NeoX vs `q_proj` in Llama). ALWAYS verify names before applying LoRA.
- **Save/Load**: `peft_model.save_pretrained()` only saves the small adapter weights. You NEED the original base model to load them back for inference.
- **Batch Size**: 4-bit models are memory-efficient but can have slower 'gradient accumulation' if batch sizes are set too high.

### Key Takeaways
- QLoRA combined with LoRA allows fine-tuning 7B+ models on a single consumer GPU (24GB VRAM).
- Rank `r=8` or `r=16` is typically sufficient for most instruction-tuning tasks.
- `device_map="auto"` automatically balances layers across multiple GPUs if available.